# 06 — EPC Recommendations Analysis

Each EPC certificate comes with a set of recommended improvements and their indicative costs. This notebook joins the recommendations to London social rented certificates to answer:

- What types of improvements are most commonly needed in London social stock?
- Which are the most expensive?
- How does this break down by borough — and do our priority boroughs (from `05_build_gold.ipynb`) face higher retrofit costs?

**Columns in recommendations data:**
- `certificate_number` — links to the EPC certificate (join key)
- `improvement_item` — priority order of the recommendation (1 = most important)
- `improvement_id` — numeric code for the improvement type
- `improvement_summary_text` — short name (e.g. 'Loft insulation', 'Solar water heating')
- `improvement_descr_text` — full description
- `indicative_cost` — estimated cost as a string range (e.g. '£800 - £1,200')

In [ ]:
import os, glob
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, countDistinct, avg, sum as spark_sum, round as spark_round,
    trim, when, lit, desc, udf, regexp_replace
)
from pyspark.sql.types import FloatType
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('recommendations') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark ready')

## 1. Load data and join to London social rented certificates

In [ ]:
BRONZE = '../data/bronze/epc_raw'
GOLD   = '../data/gold'

# Read all certificate files — filter to London social rented
cert_files = glob.glob(f'{BRONZE}/certificates-*.csv')
rec_files  = glob.glob(f'{BRONZE}/recommendations-*.csv')
print(f'Certificate files: {len(cert_files)}')
print(f'Recommendation files: {len(rec_files)}')

certs = (
    spark.read.csv(cert_files, header=True, inferSchema=False)
    .filter(
        (col('region') == 'E12000007') &
        (col('tenure') == 'rental (social)')
    )
    .select(
        col('certificate_number'),
        col('local_authority_label').alias('borough'),
        col('current_energy_rating').alias('epc_rating'),
        col('current_energy_efficiency').alias('epc_score'),
        col('construction_age_band'),
        col('property_type')
    )
)

recs = spark.read.csv(rec_files, header=True, inferSchema=False) \
    .select(
        col('certificate_number'),
        col('improvement_item').cast('int').alias('priority'),
        col('improvement_id'),
        col('improvement_summary_text').alias('improvement_type'),
        col('indicative_cost')
    )

print(f'London social certs: {certs.count():,}')
print(f'Total recommendations rows: {recs.count():,}')

In [ ]:
# Join recommendations to London social rented certs
joined = recs.join(certs, on='certificate_number', how='inner')
print(f'Joined rows (recommendations for London social stock): {joined.count():,}')

# How many certs have at least one recommendation?
certs_with_recs = joined.select('certificate_number').distinct().count()
total_certs = certs.count()
print(f'Certs with at least one recommendation: {certs_with_recs:,} ({certs_with_recs/total_certs*100:.1f}%)')

## 2. Parse indicative cost to numeric midpoint

Costs are stored as strings like '£800 - £1,200' or '£4,000 - £6,000'. We extract the midpoint for aggregation.

In [ ]:
# Show raw cost values first
display(joined.groupBy('indicative_cost').count().orderBy('count', ascending=False).limit(20).toPandas().style.format(thousands=","))

In [ ]:
def cost_midpoint(s):
    """Parse '£800 - £1,200' → 1000.0"""
    if s is None: return None
    import re
    nums = re.findall(r'[\d]+', s.replace(',', ''))
    if len(nums) == 2:
        return (float(nums[0]) + float(nums[1])) / 2
    elif len(nums) == 1:
        return float(nums[0])
    return None

def cost_low(s):
    """Parse '£800 - £1,200' → 800.0 (lower bound)"""
    if s is None: return None
    import re
    nums = re.findall(r'[\d]+', s.replace(',', ''))
    if len(nums) >= 1:
        return float(nums[0])
    return None

def cost_high(s):
    """Parse '£800 - £1,200' → 1200.0 (upper bound)"""
    if s is None: return None
    import re
    nums = re.findall(r'[\d]+', s.replace(',', ''))
    if len(nums) == 2:
        return float(nums[1])
    elif len(nums) == 1:
        return float(nums[0])
    return None

cost_mid_udf  = udf(cost_midpoint, FloatType())
cost_low_udf  = udf(cost_low,      FloatType())
cost_high_udf = udf(cost_high,     FloatType())

joined = (
    joined
    .withColumn('cost_midpoint', cost_mid_udf(col('indicative_cost')))
    .withColumn('cost_low',      cost_low_udf(col('indicative_cost')))
    .withColumn('cost_high',     cost_high_udf(col('indicative_cost')))
)

print('Sample with parsed costs (low / mid / high):')
display(joined.select('indicative_cost', 'cost_low', 'cost_midpoint', 'cost_high').distinct() \
      .orderBy('cost_midpoint').limit(15).toPandas().style.format(thousands=","))

## 3. Most common improvement types across London social stock

In [ ]:
print('=== Most commonly recommended improvements (London social rented) ===')
display(joined.groupBy('improvement_type') \
    .agg(
        count('*').alias('times_recommended'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_indicative_cost_£'),
    ) \
    .orderBy(desc('times_recommended')) \
    .limit(20).toPandas().style.format(thousands=","))

## 4. Most expensive improvement types

Cost matters for retrofit planning — some improvements are cheap and high-impact, others are expensive and unavoidable for old stock.

In [ ]:
print('=== Most expensive improvements (avg indicative cost) ===')
display(joined.groupBy('improvement_type') \
    .agg(
        count('*').alias('times_recommended'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
    ) \
    .filter(col('times_recommended') > 100) \
    .orderBy(desc('avg_cost_£')) \
    .limit(15).toPandas().style.format(thousands=","))

## 5. Borough-level retrofit cost analysis

Which boroughs face the highest estimated total and average retrofit costs?

In [ ]:
borough_costs = (
    joined
    .groupBy('borough')
    .agg(
        count('certificate_number').alias('total_recommendations'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_per_recommendation_£'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 2).alias('total_indicative_cost_£m'),
    )
    .orderBy(desc('total_indicative_cost_£m'))
)

print('=== Borough retrofit cost estimates ===')
display(borough_costs.limit(33).toPandas().style.format(thousands=","))

borough_costs.write.mode('overwrite').parquet(f'{GOLD}/borough_retrofit_costs')
print('Saved borough_retrofit_costs')

## 6. Top improvement types in our priority boroughs

Focus on the top 5 boroughs from the priority ranking. What do they actually need to fix?

In [ ]:
priority_boroughs = [
    'Barking and Dagenham', 'Haringey', 'Lambeth',
    'Hammersmith and Fulham', 'Enfield'
]

priority_recs = (
    joined
    .filter(col('borough').isin(priority_boroughs))
    .groupBy('borough', 'improvement_type')
    .agg(
        count('*').alias('times_recommended'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£')
    )
)

# Top 5 improvements per borough
w = Window.partitionBy('borough').orderBy(desc('times_recommended'))
top_per_borough = priority_recs \
    .withColumn('rnk', rank().over(w)) \
    .filter(col('rnk') <= 5) \
    .orderBy('borough', 'rnk')

print('=== Top 5 recommended improvements per priority borough ===')
display(top_per_borough.limit(50).toPandas().style.format(thousands=","))

## 7. Wall insulation — the key indicator for old stock

Solid wall insulation is the most expensive and most critical improvement for pre-1950 stock. How prevalent is it in our priority boroughs?

In [ ]:
wall_insulation_keywords = ['wall insulation', 'solid wall', 'cavity wall']

# Check how improvement_type is worded
display(joined.filter(
    col('improvement_type').rlike('(?i)wall')
).groupBy('improvement_type').count().orderBy(desc('count')).limit(10).toPandas().style.format(thousands=","))

In [ ]:
wall_flag = col('improvement_type').rlike('(?i)wall insulation')

wall_by_borough = (
    joined
    .groupBy('borough')
    .agg(
        count('certificate_number').alias('total_recs'),
        spark_sum(when(wall_flag, 1).otherwise(0)).alias('wall_insulation_recs'),
    )
    .withColumn(
        'pct_needing_wall_insulation',
        spark_round(col('wall_insulation_recs') / col('total_recs') * 100, 1)
    )
    .orderBy(desc('pct_needing_wall_insulation'))
)

print('=== % of recommendations including wall insulation by borough ===')
display(wall_by_borough.limit(33).toPandas().style.format(thousands=","))

## 8. Retrofit cost scenarios — optimistic, central, pessimistic

The indicative costs are RdSAP bands — fixed national ranges, not precise quotes.
Rather than treating the midpoint as a single estimate, we use all three bounds
to produce a cost range per borough:

- **Optimistic**: every improvement comes in at the lower bound of its cost range
- **Central**: midpoints (our existing estimate)
- **Pessimistic**: every improvement comes in at the upper bound

This gives a realistic cost corridor rather than a single figure, which is more
useful for actual budget planning — and more honest about the uncertainty in RdSAP estimates.

In [ ]:
borough_scenarios = (
    joined
    .groupBy('borough')
    .agg(
        count('certificate_number').alias('total_recommendations'),
        countDistinct('certificate_number').alias('properties_with_recs'),
        spark_round(spark_sum('cost_low')      / 1_000_000, 2).alias('total_optimistic_£m'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 2).alias('total_central_£m'),
        spark_round(spark_sum('cost_high')     / 1_000_000, 2).alias('total_pessimistic_£m'),
        spark_round(spark_sum('cost_low')      / countDistinct('certificate_number'), 0).alias('per_home_optimistic_£'),
        spark_round(spark_sum('cost_midpoint') / countDistinct('certificate_number'), 0).alias('per_home_central_£'),
        spark_round(spark_sum('cost_high')     / countDistinct('certificate_number'), 0).alias('per_home_pessimistic_£'),
        spark_round(
            (spark_sum('cost_high') - spark_sum('cost_low')) / spark_sum('cost_midpoint') * 100, 1
        ).alias('uncertainty_pct'),
    )
    .orderBy(desc('per_home_central_£'))
)

print('=== Borough retrofit cost scenarios ===')
print('Sorted by cost per home (central estimate) — shows which boroughs have most expensive stock to retrofit.')
print('Total cost (£m) shows budget required to retrofit all social rented stock in each borough.')
print()
display(borough_scenarios.select(
    'borough', 'properties_with_recs',
    'per_home_optimistic_£', 'per_home_central_£', 'per_home_pessimistic_£',
    'total_central_£m', 'uncertainty_pct'
).limit(33).toPandas().style.format(thousands=","))

borough_scenarios.write.mode('overwrite').parquet(f'{GOLD}/borough_retrofit_scenarios')
print('Saved borough_retrofit_scenarios')

In [ ]:
priority_boroughs = [
    'Barking and Dagenham', 'Haringey', 'Lambeth',
    'Hammersmith and Fulham', 'Enfield'
]

print('=== Cost scenarios for TOP 5 PRIORITY BOROUGHS ===')
print('Per-home cost shows retrofit difficulty; total cost shows budget required.')
(
    joined
    .filter(col('borough').isin(priority_boroughs))
    .groupBy('borough')
    .agg(
        countDistinct('certificate_number').alias('properties'),
        spark_round(spark_sum('cost_low')      / countDistinct('certificate_number'), 0).alias('per_home_low_£'),
        spark_round(spark_sum('cost_midpoint') / countDistinct('certificate_number'), 0).alias('per_home_mid_£'),
        spark_round(spark_sum('cost_high')     / countDistinct('certificate_number'), 0).alias('per_home_high_£'),
        spark_round(spark_sum('cost_low')      / 1_000_000, 2).alias('total_low_£m'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 2).alias('total_mid_£m'),
        spark_round(spark_sum('cost_high')     / 1_000_000, 2).alias('total_high_£m'),
    )
    .orderBy(desc('per_home_mid_£'))
).show(truncate=False)

print()
print('=== Most expensive improvement types — scenario range ===')
display(
    joined
    .groupBy('improvement_type')
    .agg(
        count('*').alias('times_recommended'),
        spark_round(avg('cost_low'),      0).alias('avg_low_£'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_mid_£'),
        spark_round(avg('cost_high'),     0).alias('avg_high_£'),
    )
    .filter(col('times_recommended') > 100)
    .orderBy(desc('avg_mid_£'))
    .limit(20).toPandas().style.format(thousands=","))

## 9. Cavity wall vs solid wall insulation — retrofit difficulty by borough

**This is the key cost differentiator.**

- **Improvement ID 6** = cavity wall insulation (~£1,500). Applies to properties built after ~1920 with a gap between inner and outer brick layers. Relatively quick and cheap.
- **Improvement ID 7** = internal or external solid wall insulation (~£8,000–£25,000). Applies to pre-1920 solid-brick and pre-1940s properties. The most expensive single retrofit measure.

A borough where most wall insulation recommendations are solid wall (ID 7) faces fundamentally more expensive retrofit than one where most are cavity (ID 6), even if both have the same % of stock below EPC C. This is a more precise retrofit difficulty signal than % pre-1950 stock, and is saved to gold for use in the clustering analysis.

In [ ]:
from pyspark.sql.functions import col, count, countDistinct, when, spark_round, desc, avg, sum as spark_sum

wall_by_borough = (
    joined
    .groupBy('borough')
    .agg(
        count('certificate_number').alias('total_recs'),
        spark_sum(when(col('improvement_id').cast('int') == 6, 1).otherwise(0)).alias('cavity_wall_recs'),
        spark_sum(when(col('improvement_id').cast('int') == 7, 1).otherwise(0)).alias('solid_wall_recs'),
    )
    .withColumn('total_wall_recs', col('cavity_wall_recs') + col('solid_wall_recs'))
    .withColumn('pct_solid_wall',
        spark_round(col('solid_wall_recs') / col('total_wall_recs') * 100, 1)
    )
    .withColumn('pct_cavity_wall',
        spark_round(col('cavity_wall_recs') / col('total_wall_recs') * 100, 1)
    )
    .filter(col('total_wall_recs') > 0)
    .orderBy(desc('pct_solid_wall'))
)

print('=== Cavity vs solid wall insulation recommendations by borough ===')
print('Higher % solid wall = more expensive and difficult retrofit stock')
display(wall_by_borough.select(
    'borough', 'total_wall_recs', 'cavity_wall_recs', 'solid_wall_recs',
    'pct_cavity_wall', 'pct_solid_wall'
).limit(33).toPandas().style.format(thousands=","))

# Save to gold — used in 07_clustering_analysis.ipynb as retrofit difficulty metric
wall_by_borough.write.mode('overwrite').parquet(f'{GOLD}/borough_wall_type')
print('Saved borough_wall_type to gold')

## 10. Cost by improvement category

The 35+ individual improvement types group into 6 meaningful categories. Viewing cost at category level is cleaner for reporting and Warm Homes Fund bids — it tells you whether a borough's retrofit cost is driven by expensive structural works (wall insulation) or cheaper mechanical upgrades (heating controls, hot water).

Categories:
- **Insulation** — wall, loft, floor, roof, party wall
- **Heating system** — boiler replacement, storage heaters, heat pumps, warm air units
- **Heating controls** — thermostats, zone controls, programmer upgrades
- **Glazing & draughts** — double glazing, secondary glazing, draught proofing, doors
- **Hot water** — cylinder insulation, cylinder thermostats
- **Renewables** — solar PV, solar thermal, wind turbine, heat recovery

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def categorise(improvement):
    if improvement is None:
        return 'Other'
    t = improvement.lower()
    if any(x in t for x in ['wall insulation', 'loft insulation', 'floor insulation',
                              'roof insulation', 'room-in-roof', 'flat roof', 'party wall']):
        return 'Insulation'
    if any(x in t for x in ['boiler', 'heating unit', 'warm air', 'storage heater',
                              'heat pump', 'biomass', 'wood pellet', 'room heater']):
        return 'Heating system'
    if any(x in t for x in ['heating control', 'zone control', 'thermostat', 'programmer',
                              'time and temperature']):
        return 'Heating controls'
    if any(x in t for x in ['glazing', 'double glaz', 'secondary glaz', 'draught', 'door']):
        return 'Glazing & draughts'
    if any(x in t for x in ['hot water cylinder', 'cylinder insulation', 'cylinder jacket',
                              'cylinder thermostat', 'immersion']):
        return 'Hot water cylinder'
    if any(x in t for x in ['solar', 'photovoltaic', 'wind turbine', 'heat recovery']):
        return 'Renewables'
    if 'lighting' in t:
        return 'Lighting'
    return 'Other'

categorise_udf = udf(categorise, StringType())
joined = joined.withColumn('category', categorise_udf(col('improvement_type')))

print('=== Retrofit cost by category — London social rented stock ===')
(
    joined
    .groupBy('category')
    .agg(
        count('*').alias('recommendations'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_per_rec_£'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 1).alias('total_cost_£m'),
        spark_round(spark_sum('cost_low')      / 1_000_000, 1).alias('total_low_£m'),
        spark_round(spark_sum('cost_high')     / 1_000_000, 1).alias('total_high_£m'),
    )
    .orderBy(desc('total_cost_£m'))
).show(truncate=False)

print('=== Cost by category — TOP 5 PRIORITY BOROUGHS ===')
priority_boroughs = ['Barking and Dagenham', 'Haringey', 'Lambeth', 'Hammersmith and Fulham', 'Enfield']
display(
    joined
    .filter(col('borough').isin(priority_boroughs))
    .groupBy('borough', 'category')
    .agg(
        count('*').alias('recs'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 2).alias('cost_£m'),
    )
    .orderBy('borough', desc('cost_£m'))
    .limit(50).toPandas().style.format(thousands=","))

## 11. Quick wins vs major works

Not all outstanding retrofit work is equally difficult or expensive. Splitting by cost tier shows whether a borough still has cheap, quick measures available (loft insulation, cylinder jackets, lighting) or whether the remaining work is almost entirely expensive structural intervention (solid wall insulation, full glazing replacement, boiler replacement).

Thresholds:
- **Quick win**: indicative cost midpoint ≤ £1,000
- **Major works**: indicative cost midpoint > £1,000

In [ ]:
joined_tiered = joined.withColumn(
    'work_tier',
    when(col('cost_midpoint') <= 1000, 'Quick win').otherwise('Major works')
)

print('=== Quick wins vs major works — London-wide ===')
(
    joined_tiered
    .groupBy('work_tier')
    .agg(
        count('*').alias('recommendations'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 1).alias('total_cost_£m'),
    )
).show(truncate=False)

print('=== Quick wins vs major works by borough (% of recommendations) ===')
display(
    joined_tiered
    .groupBy('borough')
    .agg(
        count('*').alias('total_recs'),
        spark_round(
            spark_sum(when(col('work_tier') == 'Quick win', 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_quick_wins'),
        spark_round(
            spark_sum(when(col('work_tier') == 'Major works', 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_major_works'),
        spark_round(spark_sum('cost_midpoint') / countDistinct('certificate_number'), 0).alias('cost_per_home_£'),
    )
    .orderBy(desc('pct_major_works'))
    .limit(33).toPandas().style.format(thousands=","))

print('=== Priority boroughs — quick win vs major works breakdown ===')
display(
    joined_tiered
    .filter(col('borough').isin(priority_boroughs))
    .groupBy('borough', 'work_tier', 'improvement_type')
    .agg(
        count('*').alias('recs'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
    )
    .filter(col('work_tier') == 'Quick win')
    .orderBy('borough', desc('recs'))
    .limit(40).toPandas().style.format(thousands=","))

## 12. Priority order analysis — what do assessors recommend first?

`improvement_item` is the assessor's priority rank within each certificate: 1 = single most impactful measure for that property. Analysing which improvement types most often appear at priority 1 tells you what assessors consistently identify as the biggest lever — more actionable than knowing something is merely common.

In [ ]:
print('=== Improvements most frequently ranked priority 1 (London social rented) ===')
(
    joined
    .filter(col('priority') == 1)
    .groupBy('improvement_type', 'category')
    .agg(
        count('*').alias('times_top_priority'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
    )
    .orderBy(desc('times_top_priority'))
    .limit(15)
).show(truncate=False)

print('=== Priority 1 improvement by borough — top 5 priority boroughs ===')
from pyspark.sql.window import Window
from pyspark.sql.functions import rank as spark_rank

w = Window.partitionBy('borough').orderBy(desc('times_top'))
display(
    joined
    .filter((col('priority') == 1) & col('borough').isin(priority_boroughs))
    .groupBy('borough', 'improvement_type', 'category')
    .agg(count('*').alias('times_top'))
    .withColumn('rnk', spark_rank().over(w))
    .filter(col('rnk') <= 3)
    .orderBy('borough', 'rnk')
    .limit(30).toPandas().style.format(thousands=","))

print('=== How priority order affects cost — do expensive works come first? ===')
(
    joined
    .groupBy('priority')
    .agg(
        count('*').alias('recommendations'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
    )
    .filter(col('priority') <= 10)
    .orderBy('priority')
).show(truncate=False)

## The cost of NOT retrofitting

The retrofit cost scenarios in this notebook represent the cost of action. It is worth framing these against the cost of inaction.

**MEES compliance risk:** Any property that fails to reach band C by the 2030 deadline cannot legally be let to new tenants. For a housing association, a void property that cannot be re-let is lost rental income plus ongoing maintenance liability. At average London social rents (~£120/week), a property sitting void for 6 months while awaiting retrofit costs ~£3,000 in lost income alone — often more than the cheaper improvement measures (loft insulation, cylinder jacket, heating controls) that might be all that’s needed to cross the band C threshold.

**Sequencing matters:** Not all properties need expensive structural works to reach band C. The quick wins vs major works analysis (section 11) shows what proportion of each borough’s outstanding work is under £1,000. Properties close to the band C threshold may only need one or two cheap measures — identifying these first maximises compliance at minimum cost.

**Warm Homes Fund:** The grant covers up to 100% of eligible retrofit costs for the worst-performing properties with the most deprived tenants. The cost scenarios here represent the gross cost before grant — the net cost to the association could be substantially lower for priority boroughs.

## 13. Summary

Key findings from the full recommendations analysis:

In [ ]:
print('=== RETROFIT COST SUMMARY ===')
print()

print('Top 5 most common improvements London-wide:')
joined.groupBy('improvement_type') \
    .agg(
        count('*').alias('n'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£')
    ) \
    .orderBy(desc('n')).limit(5).show(truncate=False)

print('Priority boroughs — avg cost per recommendation:')
joined.filter(col('borough').isin(priority_boroughs)) \
    .groupBy('borough') \
    .agg(spark_round(avg('cost_midpoint'), 0).alias('avg_cost_per_rec_£')) \
    .orderBy(desc('avg_cost_per_rec_£')).show(truncate=False)